# Week 7：XGBoost 调参——学习率与树数量

目标：在树深度固定为 3 时，比较几组 `learning_rate` 与 `n_estimators` 的组合。理解它们必须一起调，而不是孤立地看某一个参数。

## 先建立直觉

Boosting 会一棵接一棵地修正已有模型。`learning_rate` 决定每一棵树修正多少：

- 学习率大：每一步改得多，少量树就能拟合，但更容易走得过头。
- 学习率小：每一步改得少，通常需要更多树；训练更细致，但耗时更长。

所以常见搭配是：较小学习率配较多树。

In [1]:
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from xgboost import XGBClassifier

In [2]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data
y = dataset.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [3]:
settings = [
    {'learning_rate': 0.30, 'n_estimators': 30},
    {'learning_rate': 0.10, 'n_estimators': 100},
    {'learning_rate': 0.05, 'n_estimators': 200},
    {'learning_rate': 0.03, 'n_estimators': 350},
]

rows = []
for setting in settings:
    model = XGBClassifier(
        **setting,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        eval_metric='logloss',
        random_state=42,
        n_jobs=1,
    )
    scores = cross_validate(
        model, X_train, y_train, cv=cv, scoring='accuracy',
        return_train_score=True, n_jobs=-1,
    )
    rows.append({
        **setting,
        'train_mean': scores['train_score'].mean(),
        'validation_mean': scores['test_score'].mean(),
        'validation_std': scores['test_score'].std(),
    })

rate_results = pd.DataFrame(rows)
display(rate_results.sort_values('validation_mean', ascending=False))

,learning_rate,n_estimators,train_mean,validation_mean,validation_std
2,0.05,200,1.000000,0.973626,0.016447
3,0.03,350,1.000000,0.967033,0.015541
1,0.10,100,0.999451,0.964835,0.014579
0,0.30,30,0.998901,0.962637,0.017855


## 怎么读结果

先按 `validation_mean` 排序；如果前两名很接近，再比较 `validation_std` 和训练成本。这里无需追求唯一“标准答案”：参数选择是针对数据、预算和业务目标的实验决策。

思考题：为什么不能只把 `n_estimators` 一直加大，而不观察交叉验证结果？